In [ ]:
import pandas as pd
import calendar
import re

indent_file = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"
bom_file    = r"D:/Tushar/main_with_subs_only.xlsx"

# ───────────────────────────────────────────────
#  Read files
# ───────────────────────────────────────────────
try:
    indent_df = pd.read_excel(indent_file)
    bom_df    = pd.read_excel(bom_file)
except Exception as e:
    print("File read error:", e)
    exit()

# Clean string columns (very important)
for df in [indent_df, bom_df]:
    df.columns = df.columns.str.strip()

bom_df['Sub_Label']  = bom_df['Sub_Label'].astype(str).str.strip()
bom_df['Main_Label'] = bom_df['Main_Label'].astype(str).str.strip()

# Try to find the indent / part column automatically
possible_part_cols = [c for c in indent_df.columns if 'part' in c.lower() or 'code' in c.lower() or 'fg' in c.lower() or 'model' in c.lower()]
possible_qty_cols  = [c for c in indent_df.columns if 'qty' in c.lower() or 'indent' in c.lower() or 'plan' in c.lower()]

print("Possible part columns in indent file:", possible_part_cols)
print("Possible qty columns in indent file :", possible_qty_cols)

# ───────────────────────────────────────────────
#  Month / days detection
# ───────────────────────────────────────────────
pattern = re.compile(r"([A-Za-z]{3})'(\d{2})", re.I)
month_cols = [c for c in indent_df.columns if pattern.search(str(c))]

if not month_cols:
    print("No month columns like 'Jan'22' found.")
    print("All columns:", list(indent_df.columns))
    exit()

latest_col = month_cols[-1]
print("\nDetected latest month column →", latest_col)

match = pattern.search(latest_col)
if not match:
    print("Cannot parse month from:", latest_col)
    exit()

month_str = match.group(1).title()
year_suffix = match.group(2)
year = 2000 + int(year_suffix)
try:
    month_num = list(calendar.month_abbr).index(month_str)
except ValueError:
    print("Invalid month abbreviation:", month_str)
    exit()

days_in_month = calendar.monthrange(year, month_num)[1]
print(f"→ {month_str} {year} → {days_in_month} days\n")

# ───────────────────────────────────────────────
#  Prepare indent data
# ───────────────────────────────────────────────
part_col = 'Part number'   # ← change here if name is different

if part_col not in indent_df.columns:
    print(f"Column '{part_col}' not found. Available:", list(indent_df.columns))
    exit()

indent_df = indent_df[[part_col, latest_col]].copy()
indent_df = indent_df.dropna(subset=[latest_col])           # drop rows without qty
indent_df[part_col] = indent_df[part_col].astype(str).str.strip()

# Make sure qty is numeric
indent_df[latest_col] = pd.to_numeric(indent_df[latest_col], errors='coerce')
indent_df = indent_df.dropna(subset=[latest_col])

indent_df = indent_df.rename(columns={part_col: 'Switch', latest_col: 'Monthly_Qty'})
indent_df['Daily_Qty'] = indent_df['Monthly_Qty'] / days_in_month

print("Indent summary:")
print(indent_df.head(8))
print(f"→ {len(indent_df)} switch types with plan\n")

# ───────────────────────────────────────────────
#  Prepare BOM
# ───────────────────────────────────────────────
bom_df = bom_df[['Main_Label', 'Sub_Label', 'Sub_Count']].copy()
bom_df = bom_df.rename(columns={'Main_Label': 'Child', 'Sub_Label': 'Switch', 'Sub_Count': 'Usage_Qty'})

bom_df['Switch'] = bom_df['Switch'].astype(str).str.strip()
bom_df['Child']  = bom_df['Child'].astype(str).str.strip()

print(f"BOM has {len(bom_df):,} lines\n")

# ───────────────────────────────────────────────
#  Merge + explode
# ───────────────────────────────────────────────
merged = pd.merge(
    bom_df,
    indent_df[['Switch', 'Daily_Qty']],
    on='Switch',
    how='inner'
)

print(f"After merge: {len(merged):,} matching lines  (if 0 → codes don't match!)")

if len(merged) == 0:
    print("\nSample BOM switches:", bom_df['Switch'].unique()[:8].tolist())
    print("Sample indent switches:", indent_df['Switch'].unique()[:8].tolist())
    print("\n→ Check for differences in formatting, prefixes, zeros, spaces...\n")
    exit()

merged['Daily_Child_Need'] = merged['Daily_Qty'] * merged['Usage_Qty']

# Aggregate
result = merged.groupby('Child', as_index=False)['Daily_Child_Need'].sum()
result = result.rename(columns={'Daily_Child_Need': 'Daily_Qty'})

result['Two_Day_Qty'] = result['Daily_Qty'] * 2

# Sort most critical first
result = result.sort_values('Two_Day_Qty', ascending=False).round({'Daily_Qty': 2, 'Two_Day_Qty': 1})

print("\nTop 12 results:")
print(result.head(12))

# Save
output_file = "Two_Day_Child_Qty.xlsx"
result.to_excel(output_file, index=False)
print(f"\nSaved → {output_file}")